In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType, TimestampType

# COMMAND ----------
# Canonical schema definition: (column_name, data_type)
canonical_cols = [
    ("VendorID", IntegerType()),
    ("tpep_pickup_datetime", TimestampType()),
    ("tpep_dropoff_datetime", TimestampType()),
    ("passenger_count", DoubleType()),
    ("trip_distance", DoubleType()),
    ("RatecodeID", DoubleType()),
    ("store_and_fwd_flag", StringType()),
    ("PULocationID", IntegerType()),
    ("DOLocationID", IntegerType()),
    ("payment_type", IntegerType()),
    ("fare_amount", DoubleType()),
    ("extra", DoubleType()),
    ("mta_tax", DoubleType()),
    ("tip_amount", DoubleType()),
    ("tolls_amount", DoubleType()),
    ("improvement_surcharge", DoubleType()),
    ("total_amount", DoubleType()),
    ("congestion_surcharge", DoubleType()),
    ("airport_fee", DoubleType()),
]

source_cols_set = set(raw_df.columns)

# COMMAND ----------
# Helper strings and functions
TS_FORMAT = "MM/dd/yyyy hh:mm:ss a"

def clean_numeric(col):
    """
    Defensive numeric cleaning:
    - trim whitespace
    - remove thousands separators (commas) e.g. 1,000.55 or 299,899,990.00
    - convert empty strings to NULL
    - cast to DOUBLE
    """
    c = F.trim(col)
    c = F.when((c == "") | c.isNull(), F.lit(None)).otherwise(c)
    c = F.regexp_replace(c, ",", "")
    return c.cast(DoubleType())

def clean_timestamp(col):
    """
    Defensive timestamp parsing:
    - trim whitespace
    - convert empty strings to NULL
    - parse using explicit format
    """
    c = F.trim(col)
    c = F.when((c == "") | c.isNull(), F.lit(None)).otherwise(c)
    return F.to_timestamp(c, TS_FORMAT)

def safe_cast(col, dtype):
    """
    Applies the appropriate defensive conversion for each target type.
    """
    if isinstance(dtype, TimestampType):
        return clean_timestamp(col)
    if isinstance(dtype, DoubleType):
        return clean_numeric(col)
    # For Int/String, do trim + empty->NULL then cast
    c = F.trim(col)
    c = F.when((c == "") | c.isNull(), F.lit(None)).otherwise(c)
    return c.cast(dtype)

# COMMAND ----------
# Build select expressions with defensive casting + add missing cols as NULL
select_exprs = []
for col_name, dtype in canonical_cols:
    if col_name in source_cols_set:
        select_exprs.append(safe_cast(F.col(col_name), dtype).alias(col_name))
    else:
        select_exprs.append(F.lit(None).cast(dtype).alias(col_name))

bronze_df = (
    raw_df
    .select(*select_exprs)
    .withColumn("ingest_ts", F.current_timestamp())
)

display(bronze_df.limit(10))
print("Bronze columns:", len(bronze_df.columns))


Verify

In [0]:
# COMMAND ----------
from pyspark.sql import functions as F

# Check parsing success for timestamps
bronze_df.select(
    F.count(F.when(F.col("tpep_pickup_datetime").isNull(), 1)).alias("null_pickup_ts"),
    F.count(F.when(F.col("tpep_dropoff_datetime").isNull(), 1)).alias("null_dropoff_ts")
).show()

# Check key numeric columns for nulls (after casting)
key_numeric = ["passenger_count", "trip_distance", "RatecodeID", "fare_amount", "total_amount", "tip_amount"]
exprs = [F.count(F.when(F.col(c).isNull(), 1)).alias(f"null_{c}") for c in key_numeric]
bronze_df.select(*exprs).show()

# Inspect suspicious ranges (comparing to the expected values from documentation)
bronze_df.selectExpr(
    "min(passenger_count) as min_passenger_count",
    "max(passenger_count) as max_passenger_count",
    "min(RatecodeID) as min_RatecodeID",
    "max(RatecodeID) as max_RatecodeID",
    "min(trip_distance) as min_trip_distance",
    "max(trip_distance) as max_trip_distance"
).show()

# Quick distribution checks
bronze_df.groupBy("RatecodeID").count().orderBy("RatecodeID").show(50, truncate=False)
bronze_df.groupBy("passenger_count").count().orderBy("passenger_count").show(50, truncate=False)
bronze_df.groupBy("payment_type").count().orderBy("payment_type").show(50, truncate=False)


In [0]:
bronze_df.selectExpr(
    "min(trip_distance)",
    "max(trip_distance)"
).show()


## Basic initial check for any outliers

In [0]:
bronze_df.selectExpr("min(trip_distance)", "min(fare_amount)", "min(total_amount)").show()
bronze_df.selectExpr("max(trip_distance)", "max(fare_amount)", "max(total_amount)").show()